In [1]:
# ==============================================================
# Sales Forecast v3
# - Auto Exog Selection via Validation (forward-style)
# - Y-Models: RF, XGB, Y-Ensemble (RF+XGB) with inverse-MAE weights
# - Bootstrap PIs (80% / 95%) using VAL residual bootstrap
# - SHAP / Feature Importances (fallback if shap missing)
# - Stockout Probability for next 3 / 6 months (bootstrap)
# - Exports CSVs & PNGs under outputs/
# ==============================================================

import warnings, os, sys, logging
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
#warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Maximum Likelihood optimization failed to converge")
warnings.filterwarnings("ignore", message="Optimization failed to converge")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("statsmodels").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Optional SHAP
try:
    import shap
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ------------------ Config ------------------
CSV_PATH   = "veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv"

VAL_START  = pd.Timestamp("2024-07-01")
VAL_END    = pd.Timestamp("2024-12-01")
TEST_START = pd.Timestamp("2025-01-01")
TEST_END   = pd.Timestamp("2025-07-01")
TEST_END_SHORT = pd.Timestamp("2025-03-01")  # 3 months

RANDOM_STATE = 42
EXOG_VAL_H   = 6
EPS_PROPHET  = 0.05
B_BOOT       = 800   # bootstrap sim count
SEED         = 1337  # bootstrap seed for reproducibility

FEATURES_Y = [
    "orders","stock",
    "orders_lag1","orders_lag3",
    "stock_lag1","stock_lag3",
    "y_lag1",
    "orders_ratio",
    "month","year",
]

# Allow override of starting stock for stockout simulation; None -> infer from data
STARTING_STOCK_OVERRIDE = None

# ------------------ Utils ------------------
rng_boot = np.random.default_rng(SEED)

def mae_rmse_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    denom = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    return mae, rmse, mape

def ensure_ms_freq(df):
    d = df.copy().sort_values("ds")
    d["ds"] = d["ds"].dt.to_period("M").dt.to_timestamp(how="start")
    d = d.set_index("ds").sort_index()
    d.index = pd.DatetimeIndex(d.index, freq="MS")
    return d.reset_index()

def add_calendar(df):
    d = df.copy()
    d["year"]  = d["ds"].dt.year
    d["month"] = d["ds"].dt.month
    return d

def rolling_impute(s, causal=False):
    x = pd.to_numeric(s, errors="coerce")
    if causal:
        x = x.ffill()
        x = x.rolling(window=3, min_periods=1).mean()
        x = x.bfill()
    else:
        roll = x.rolling(window=3, center=True, min_periods=1).mean()
        x = x.where(~x.isna(), roll).ffill().bfill()
    return x

def smooth_causal_ma(s, window=3):
    x = pd.to_numeric(s, errors="coerce").ffill()
    return x.rolling(window=window, min_periods=1).mean().bfill()

def winsorize_series(s, lower_q=0.05, upper_q=0.95):
    x = pd.to_numeric(s, errors="coerce")
    lo = np.nanpercentile(x, lower_q*100)
    hi = np.nanpercentile(x, upper_q*100)
    return x.clip(lo, hi)

def nonneg(s):
    return pd.to_numeric(s, errors="coerce").clip(lower=0.0)

def build_lags_y(df):
    d = df.copy()
    if "orders" in d.columns and "stock" in d.columns:
        d["orders_ratio"] = d["orders"] / d["stock"].replace(0, np.nan)
    if "y" in d.columns:
        d["y_lag1"] = d["y"].shift(1)
    if "orders" in d.columns:
        d["orders_lag1"] = d["orders"].shift(1)
        d["orders_lag3"] = d["orders"].shift(3)
    if "stock" in d.columns:
        d["stock_lag1"] = d["stock"].shift(1)
        d["stock_lag3"] = d["stock"].shift(3)
    return d

def prep_features_y(df_in, causal=False):
    d = add_calendar(df_in)
    d = build_lags_y(d)
    for col in ["orders","stock"]:
        if col in d.columns:
            d[col] = rolling_impute(d[col], causal=causal)
    for col in ["orders_lag1","orders_lag3","stock_lag1","stock_lag3","y_lag1","orders_ratio"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce").ffill().bfill().fillna(0.0)
    for c in FEATURES_Y:
        if c not in d.columns:
            d[c] = 0.0
    return d.replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------ Univariate exog (Prophet/SARIMA/ETS) ------------------
def fit_prophet(train_df, value_col):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m.fit(train_df.rename(columns={value_col:"y"}))
    return m

def forecast_prophet(model, steps):
    fut = model.make_future_dataframe(periods=steps, freq="MS")
    fc  = model.predict(fut)[["ds","yhat"]].tail(steps)
    return fc.rename(columns={"yhat":"yhat"})

def sarima_fit_best(y, p_range=(0,3), q_range=(0,3), P_range=(0,1), Q_range=(0,1)):
    best, best_aic = None, np.inf
    for p in range(p_range[0], p_range[1]+1):
        for q in range(q_range[0], q_range[1]+1):
            for P in range(P_range[0], P_range[1]+1):
                for Q in range(Q_range[0], Q_range[1]+1):
                    try:
                        r = SARIMAX(y, order=(p,1,q), seasonal_order=(P,1,Q,12),
                                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                        if r.aic < best_aic:
                            best_aic = r.aic
                            best = ((p,1,q),(P,1,Q,12))
                    except Exception:
                        pass
    return best

def fit_sarima(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best = sarima_fit_best(y)
    if best is None:
        return None
    return SARIMAX(y, order=best[0], seasonal_order=best[1],
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

def forecast_sarima(model, steps, future_idx):
    pred = model.get_forecast(steps=steps).predicted_mean
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def fit_ets(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best_model, best_aic = None, np.inf
    for trend in ["add", "mul", None]:
        for seasonal in ["add", "mul", None]:
            try:
                for damped in [True, False]:
                    if seasonal is None:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=None, damped_trend=damped).fit(optimized=True)
                    else:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=seasonal, seasonal_periods=12,
                                                     damped_trend=damped).fit(optimized=True)
                    aic = getattr(model, "aic", np.inf)
                    if aic < best_aic:
                        best_aic, best_model = aic, model
            except Exception:
                continue
    if best_model is None:
        best_model = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=12).fit(optimized=True)
    return best_model

def forecast_ets(model, steps, future_idx):
    pred = model.forecast(steps)
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def backtest_mae(series_df, value_col, method, cutoff, val_h=EXOG_VAL_H):
    s_full = series_df[["ds", value_col]].dropna().sort_values("ds")
    s = s_full[s_full["ds"] < cutoff]
    if len(s) < val_h + 6:
        return np.inf
    cut = s["ds"].max() - pd.DateOffset(months=val_h-1)
    train = s[s["ds"] < cut]
    val   = s[s["ds"] >= cut]
    steps = len(val)
    try:
        if method == "prophet":
            m = fit_prophet(train, value_col)
            yhat = forecast_prophet(m, steps)["yhat"].values
        elif method == "sarima":
            m = fit_sarima(train, value_col)
            if m is None: return np.inf
            yhat = forecast_sarima(m, steps, val["ds"].values)["yhat"].values
        else:
            m = fit_ets(train, value_col)
            yhat = forecast_ets(m, steps, val["ds"].values)["yhat"].values
    except Exception:
        return np.inf
    return mae_rmse_mape(val[value_col].values, yhat)[0]

def _postprocess_exog(df):
    d = df.copy()
    for c in ["orders","stock"]:
        if c in d.columns:
            d[c] = smooth_causal_ma(d[c], window=3)
            d[c] = winsorize_series(d[c], 0.05, 0.95)
            d[c] = nonneg(d[c])
    return d

def build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff, eps_prophet=EPS_PROPHET):
    """Inverse-MAE ensemble for orders & stock; training strictly < cutoff, forecast start..end."""
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s = df_all[["ds", col]].dropna().sort_values("ds")
        s = s[s["ds"] < cutoff]
        if s.empty:
            out[col] = 0.0; continue
        steps = len(future_idx)
        mae_p = backtest_mae(s, col, "prophet", cutoff)
        mae_s = backtest_mae(s, col, "sarima",  cutoff)
        mae_e = backtest_mae(s, col, "ets",     cutoff)
        try:
            mp  = fit_prophet(s, col); fcp = forecast_prophet(mp, steps).rename(columns={"yhat":"p"})
        except Exception: fcp = pd.DataFrame({"ds": future_idx, "p": np.nan})
        try:
            ms  = fit_sarima(s, col);  fcs = forecast_sarima(ms, steps, future_idx).rename(columns={"yhat":"s"})
        except Exception: fcs = pd.DataFrame({"ds": future_idx, "s": np.nan})
        try:
            me  = fit_ets(s, col);     fce = forecast_ets(me, steps, future_idx).rename(columns={"yhat":"e"})
        except Exception: fce = pd.DataFrame({"ds": future_idx, "e": np.nan})

        maes = np.array([mae_p, mae_s, mae_e], dtype=float)
        maes = np.where(~np.isfinite(maes) | (maes <= 0), np.nan, maes)
        inv  = 1.0 / np.where(np.isnan(maes), np.inf, maes)
        if np.isfinite(inv).sum() == 0:
            wp, ws, we = 0.60, 0.25, 0.15
        else:
            w = inv / inv.sum()
            w[0] += eps_prophet
            w = w / w.sum()
            wp, ws, we = w[0], w[1], w[2]

        tmp = (pd.DataFrame({"ds": future_idx})
                 .merge(fcp, on="ds", how="left")
                 .merge(fcs, on="ds", how="left")
                 .merge(fce, on="ds", how="left"))
        tmp[col] = (wp*tmp["p"] + ws*tmp["s"] + we*tmp["e"])
        out = out.merge(tmp[["ds",col]], on="ds", how="left")
    return _postprocess_exog(out)

# ------------------ ML-Exog (orders/stock) ------------------
EXOG_FEATURES = ["lag1","lag3","month","year"]

def make_exog_feature_frame(df_var, var_col):
    d = df_var[["ds", var_col]].sort_values("ds").copy()
    d["lag1"] = d[var_col].shift(1)
    d["lag3"] = d[var_col].shift(3)
    d["month"] = d["ds"].dt.month
    d["year"]  = d["ds"].dt.year
    return d

def train_exog_rf(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty:
        return None
    X = d[EXOG_FEATURES]; y = d[var_col]
    rf = RandomForestRegressor(
        n_estimators=400, max_depth=8, min_samples_split=2, min_samples_leaf=1,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X, y)
    return rf

def train_exog_xgb(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty:
        return None
    X = d[EXOG_FEATURES].to_numpy(); y = d[var_col].to_numpy()
    xgb = XGBRegressor(
        n_estimators=500, learning_rate=0.08, max_depth=3,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.2,
        random_state=RANDOM_STATE
    )
    xgb.fit(X, y, verbose=False)
    return xgb

def recursive_forecast_exog(model, hist_df, var_col, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    full = hist_df[["ds", var_col]].sort_values("ds").copy()
    out = []
    for ds in future_idx:
        tmp = make_exog_feature_frame(full, var_col)
        row = tmp[tmp["ds"] == ds][EXOG_FEATURES]
        if row.empty:
            row = pd.DataFrame({
                "lag1": [full[var_col].iloc[-1] if len(full) else 0.0],
                "lag3": [full[var_col].iloc[-3] if len(full) >= 3 else (full[var_col].iloc[-1] if len(full) else 0.0)],
                "month":[ds.month], "year":[ds.year]
            })
        x = row.to_numpy()
        yhat = model.predict(x)[0]
        out.append({"ds": ds, var_col: float(max(0.0, yhat))})
        full = pd.concat([full, pd.DataFrame({"ds":[ds], var_col:[yhat]})], ignore_index=True)
    return pd.DataFrame(out)

def build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, learner="rf"):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s_hist = df_all[df_all["ds"] < cutoff][["ds", col]].copy()
        if s_hist.empty:
            out[col] = 0.0; continue
        mdl = train_exog_rf(df_all[["ds", col]].copy(), col, cutoff) if learner=="rf" \
              else train_exog_xgb(df_all[["ds", col]].copy(), col, cutoff)
        if mdl is None:
            out[col] = 0.0; continue
        fc = recursive_forecast_exog(mdl, s_hist, col, start_ds, end_ds)
        out = out.merge(fc, on="ds", how="left")
    return _postprocess_exog(out)

# ------------------ ROCV on Y (same as v2) ------------------
def rolling_origin_splits(df, n_splits=3, min_train_months=24):
    d = df.sort_values("ds").copy()
    if d["ds"].nunique() < (min_train_months + n_splits):
        yield (d[d["ds"] < d["ds"].max() - pd.DateOffset(months=3)],
               d[d["ds"] >= d["ds"].max() - pd.DateOffset(months=3)])
        return
    for k in range(n_splits, 0, -1):
        val_end = d["ds"].max() - pd.DateOffset(months=k-1)
        val_start = val_end - pd.DateOffset(months=2)
        train_end = val_start - pd.DateOffset(days=1)
        train = d[(d["ds"] <= train_end)]
        val   = d[(d["ds"] >= val_start) & (d["ds"] <= val_end)]
        if len(train) >= min_train_months and len(val) >= 2:
            yield train, val

def optimize_rf_rocv(df_trainval):
    base_grid = {
        "n_estimators": [400, 700],
        "max_depth": [None, 8, 12],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
            mdl.fit(tr[FEATURES_Y], tr["y"])
            pred = mdl.predict(va[FEATURES_Y])
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best).fit(df_trainval[FEATURES_Y], df_trainval["y"])
    return final, best, best_score

def optimize_xgb_rocv(df_trainval):
    base_grid = {
        "n_estimators": [500, 800],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1.0, 2.0],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = XGBRegressor(random_state=RANDOM_STATE, **params)
            mdl.fit(tr[FEATURES_Y].to_numpy(), tr["y"].to_numpy(), verbose=False)
            pred = mdl.predict(va[FEATURES_Y].to_numpy())
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = XGBRegressor(random_state=RANDOM_STATE, **best)
    final.fit(df_trainval[FEATURES_Y].to_numpy(), df_trainval["y"].to_numpy(), verbose=False)
    return final, best, best_score

# ------------------ Y recursive forward ------------------
def recursive_forward_predict_y(model, x_cols, hist_df, future_exog, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    future_part = pd.DataFrame({"ds": future_idx}).merge(future_exog, on="ds", how="left")
    full = pd.concat([hist_df, future_part], ignore_index=True).sort_values("ds")

    preds = []
    for ds in future_idx:
        tmp = prep_features_y(full.copy(), causal=True)
        row = tmp.loc[tmp["ds"] == ds].copy()
        X = row[x_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
        y_hat = float(model.predict(X)[0])
        preds.append({"ds": ds, "yhat": y_hat})
        full.loc[full["ds"] == ds, "y"] = y_hat
        for c in ["orders","stock"]:
            if c in full.columns:
                full[c] = rolling_impute(full[c], causal=True)
    preds_df = pd.DataFrame(preds)
    used_future = full.loc[full["ds"].isin(future_idx)].copy()
    return preds_df, used_future

# ------------------ Load & prepare Y-data ------------------
os.makedirs("outputs", exist_ok=True)
os.makedirs("logs", exist_ok=True)

df_raw = pd.read_csv(CSV_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
for c in ["y","orders","stock"]:
    if c in df_raw.columns: df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
df_raw = ensure_ms_freq(df_raw)

# splits
mask_train = (df_raw["ds"] < VAL_START)
mask_val   = (df_raw["ds"] >= VAL_START) & (df_raw["ds"] <= VAL_END)

train_df = prep_features_y(df_raw.loc[mask_train].copy(), causal=False)
val_df   = prep_features_y(df_raw.loc[mask_val].copy(),   causal=True)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

# Fit Y models with ROCV
rf_model,  rf_params,  rf_rocv_mae  = optimize_rf_rocv(trainval_df)
xgb_model, xgb_params, xgb_rocv_mae = optimize_xgb_rocv(trainval_df)

print("\n=== ROCV Best Params ===")
print("RF :", rf_params,  f"| ROCV_MAE={rf_rocv_mae:.2f}")
print("XGB:", xgb_params, f"| ROCV_MAE={xgb_rocv_mae:.2f}")

# ------------------ Helper: build exog for any period ------------------
def build_exog(method, df_all, start_ds, end_ds, cutoff):
    if method == "Ensemble":
        return build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff)
    elif method == "ML-Exog RF":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, learner="rf")
    elif method == "ML-Exog XGB":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, learner="xgb")
    else:
        raise ValueError("Unknown exog method")

EXOG_METHODS = ["Ensemble", "ML-Exog RF", "ML-Exog XGB"]

# ------------------ Validation-based Exog Selection ------------------
def y_ensemble_weights(y_true, yhat_rf, yhat_xgb, eps=1e-6):
    mae_rf  = mean_absolute_error(y_true, yhat_rf)
    mae_xgb = mean_absolute_error(y_true, yhat_xgb)
    w_rf, w_xgb = 1.0/(mae_rf+eps), 1.0/(mae_xgb+eps)
    s = w_rf + w_xgb
    return w_rf/s, w_xgb/s, mae_rf, mae_xgb

def validate_exog_method(method, df_all, rf_model, xgb_model):
    # Build exog for VAL period using history < VAL_START
    exog_val = build_exog(method, df_all, VAL_START, VAL_END, cutoff=VAL_START)
    # History up to train (strict)
    hist_for_val = train_df[["ds","y","orders","stock","month","year"]].copy()

    # Predict with RF & XGB
    pred_rf , _ = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_for_val.copy(), exog_val, VAL_START, VAL_END)
    pred_xgb, _ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_for_val.copy(), exog_val, VAL_START, VAL_END)

    val_truth = val_df[["ds","y"]].copy()
    join = val_truth.merge(pred_rf, on="ds", how="left").rename(columns={"yhat":"yhat_rf"}) \
                    .merge(pred_xgb, on="ds", how="left").rename(columns={"yhat":"yhat_xgb"})
    # Weights from VAL
    w_rf, w_xgb, mae_rf, mae_xgb = y_ensemble_weights(join["y"].values, join["yhat_rf"].values, join["yhat_xgb"].values)
    yhat_ens = w_rf*join["yhat_rf"].values + w_xgb*join["yhat_xgb"].values
    mae_ens, rmse_ens, mape_ens = mae_rmse_mape(join["y"].values, yhat_ens)

    # residuals for bootstrap per variant
    res_rf  = (join["y"].values - join["yhat_rf"].values)
    res_xgb = (join["y"].values - join["yhat_xgb"].values)
    res_ens = (join["y"].values - yhat_ens)

    out = {
        "method": method,
        "weights": (w_rf, w_xgb),
        "mae_rf": mae_rf, "mae_xgb": mae_xgb,
        "mae_ens": mae_ens, "rmse_ens": rmse_ens, "mape_ens": mape_ens,
        "residuals": {"RF": res_rf, "XGB": res_xgb, "ENS": res_ens},
        "preds_val": join
    }
    return out

def select_best_exog(df_all, rf_model, xgb_model):
    rows = []
    reports = {}
    for m in EXOG_METHODS:
        rep = validate_exog_method(m, df_all, rf_model, xgb_model)
        reports[m] = rep
        rows.append([m, rep["mae_rf"], rep["mae_xgb"], rep["mae_ens"], rep["rmse_ens"], rep["mape_ens"], rep["weights"][0], rep["weights"][1]])
    tab = pd.DataFrame(rows, columns=["Exog","VAL_MAE_RF","VAL_MAE_XGB","VAL_MAE_YENS","VAL_RMSE_YENS","VAL_MAPE_YENS","w_RF","w_XGB"])
    tab.to_csv("outputs/val_exog_selection.csv", index=False)
    # choose by best Y-ENSEMBLE MAE (you can change to best single-model if desired)
    best = tab.loc[tab["VAL_MAE_YENS"].idxmin(), "Exog"]
    return best, tab, reports

best_exog, sel_table, sel_reports = select_best_exog(df_raw[["ds","orders","stock"]].copy(), rf_model, xgb_model)
print("\n=== Exog Selection (by VAL MAE of Y-ENS) ===")
print(sel_table.to_string(index=False))
print(f"\nSelected Exog: {best_exog} | w_RF={sel_reports[best_exog]['weights'][0]:.2f}, w_XGB={sel_reports[best_exog]['weights'][1]:.2f}")

# ------------------ Predict on TEST with chosen exog ------------------
def predict_with_variant(exog_table, variant, weights, hist_df, start_ds, end_ds):
    if variant == "RF":
        p,_ = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_df.copy(), exog_table, start_ds, end_ds)
        resids_val = sel_reports[best_exog]["residuals"]["RF"]
    elif variant == "XGB":
        p,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_df.copy(), exog_table, start_ds, end_ds)
        resids_val = sel_reports[best_exog]["residuals"]["XGB"]
    elif variant == "Y-ENS":
        # point: weighted average of RF + XGB per step
        future_idx = pd.date_range(start_ds, end_ds, freq="MS")
        prf,_  = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_df.copy(), exog_table, start_ds, end_ds)
        pxgb,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_df.copy(), exog_table, start_ds, end_ds)
        p = prf.merge(pxgb, on="ds", suffixes=("_rf","_xgb"))
        p["yhat"] = weights[0]*p["yhat_rf"] + weights[1]*p["yhat_xgb"]
        p = p[["ds","yhat"]]
        resids_val = sel_reports[best_exog]["residuals"]["ENS"]
    else:
        raise ValueError("Unknown variant")
    return p, np.array(resids_val, dtype=float)

def add_bootstrap_intervals(pred_df, residuals, level_low=0.10, level_high=0.90, B=B_BOOT):
    # iid bootstrap from VAL residuals; add to point-forecast to simulate predictive distribution
    if residuals.size == 0:
        # fallback to Laplace noise scaled by predicted median abs dev
        scale = max(1e-6, np.median(np.abs(pred_df["yhat"].values)) * 0.1)
        sims = pred_df["yhat"].to_numpy().reshape(-1,1) + rng_boot.laplace(0.0, scale, size=(len(pred_df), B))
    else:
        idx = rng_boot.integers(0, residuals.size, size=(len(pred_df), B))
        noise = residuals[idx]
        sims = pred_df["yhat"].to_numpy().reshape(-1,1) + noise
    lo80 = np.nanquantile(sims, 0.10, axis=1)
    hi80 = np.nanquantile(sims, 0.90, axis=1)
    lo95 = np.nanquantile(sims, 0.025, axis=1)
    hi95 = np.nanquantile(sims, 0.975, axis=1)
    out = pred_df.copy()
    out["pi80_lo"] = lo80; out["pi80_hi"] = hi80
    out["pi95_lo"] = lo95; out["pi95_hi"] = hi95
    return out, sims

def infer_starting_stock(df_raw, test_start, override=None):
    if override is not None:
        return float(override)
    prev = df_raw[df_raw["ds"] < test_start].tail(1)
    if "stock" in prev.columns and len(prev):
        val = float(prev["stock"].iloc[0])
        return max(0.0, val)
    return 0.0

def stockout_probability(starting_stock, sims_matrix):
    """
    starting_stock: scalar
    sims_matrix: shape (T, B) monthly simulated demand (>=0)
    Returns:
      p_3m, p_6m, expected_time_to_stockout_months (np.nan if never)
    """
    sims = np.maximum(0.0, sims_matrix)  # demand >= 0
    cum = np.cumsum(sims, axis=0)        # T x B
    T = sims.shape[0]
    # time-to-stockout index per path
    tts = np.full(sims.shape[1], np.nan)
    for b in range(sims.shape[1]):
        idx = np.where(cum[:,b] >= starting_stock)[0]
        if idx.size > 0:
            tts[b] = idx[0] + 1  # month index (1..T)
    # probabilities
    p3 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 3)
    p6 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 6)
    exp_t = np.nanmean(tts) if np.any(~np.isnan(tts)) else np.nan
    return float(p3), float(p6), (None if np.isnan(exp_t) else float(exp_t))

# ------------------ Build chosen exog for TEST Full & Short ------------------
exog_test_full  = build_exog(best_exog, df_raw[["ds","orders","stock"]], TEST_START, TEST_END,        cutoff=TEST_START)
exog_test_short = build_exog(best_exog, df_raw[["ds","orders","stock"]], TEST_START, TEST_END_SHORT,  cutoff=TEST_START)

# Chosen weights from VAL (for Y-ENS)
w_rf, w_xgb = sel_reports[best_exog]["weights"]
hist_min = pd.concat([train_df, val_df], ignore_index=True)[["ds","y","orders","stock","month","year"]].copy()

# Variants to run
VARIANTS = ["RF", "XGB", "Y-ENS"]

results_meta = []

for tag, (exog_tbl, end_ds) in {
    "Full":   (exog_test_full,  TEST_END),
    "Short3": (exog_test_short, TEST_END_SHORT)
}.items():
    for variant in VARIANTS:
        preds, resids = predict_with_variant(exog_tbl, variant, (w_rf, w_xgb), hist_min, TEST_START, end_ds)
        preds_pi, sims = add_bootstrap_intervals(preds, resids, B=B_BOOT)

        # Merge truth for metrics if available
        truth = df_raw[(df_raw["ds"] >= TEST_START) & (df_raw["ds"] <= end_ds)][["ds","y"]].copy()
        eval_df = truth.merge(preds_pi, on="ds", how="left")
        mae, rmse, mape = mae_rmse_mape(eval_df["y"], eval_df["yhat"])

        # Stockout probabilities on this horizon (using this variant's sims)
        starting_stock = infer_starting_stock(df_raw, TEST_START, STARTING_STOCK_OVERRIDE)
        p3, p6, exp_t = stockout_probability(starting_stock, sims)

        # save
        out_path = f"outputs/preds_{tag}_{variant.replace('-','').lower()}.csv"
        eval_df.to_csv(out_path, index=False)

        results_meta.append([tag, best_exog, variant, mae, rmse, mape, w_rf if variant=="Y-ENS" else np.nan, w_xgb if variant=="Y-ENS" else np.nan, p3, p6, exp_t])

# Summary table
summary_cols = ["Horizon","Selected_Exog","Y-Variant","MAE","RMSE","MAPE","w_RF","w_XGB","P_stockout_3m","P_stockout_6m","E_T_stockout_mo"]
summary_df = pd.DataFrame(results_meta, columns=summary_cols).sort_values(["Horizon","Y-Variant"])
summary_df.to_csv("outputs/test_summary_selected_exog.csv", index=False)

print("\n=== TEST Summary (Selected Exog) ===")
print(summary_df.to_string(index=False))

# ------------------ SHAP / Feature Importances ------------------
def dump_feature_importances(model, model_name, X_df):
    out = pd.DataFrame({"feature": X_df.columns, "importance": getattr(model, "feature_importances_", np.zeros(X_df.shape[1]))})
    out = out.sort_values("importance", ascending=False)
    out.to_csv(f"outputs/feat_importance_{model_name}.csv", index=False)
    return out

def dump_shap_values(model, model_name, X_df):
    if not HAVE_SHAP:
        return None
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_df)
        # If list (XGB multioutput), take as array
        sv_arr = np.array(sv)
        if sv_arr.ndim == 3:  # (classes, n, f)
            sv_arr = np.mean(np.abs(sv_arr), axis=0)
        mean_abs = np.mean(np.abs(sv_arr), axis=0)
        out = pd.DataFrame({"feature": X_df.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
        out.to_csv(f"outputs/shap_importance_{model_name}.csv", index=False)
        return out
    except Exception:
        return None

X_trainval = trainval_df[FEATURES_Y].copy()
fi_rf  = dump_feature_importances(rf_model,  "rf",  X_trainval)
fi_xgb = dump_feature_importances(xgb_model, "xgb", X_trainval)
shap_rf  = dump_shap_values(rf_model,  "rf",  X_trainval)
shap_xgb = dump_shap_values(xgb_model, "xgb", X_trainval)

# ------------------ Plots ------------------
def plot_with_pi(eval_df, title, savepath):
    plt.figure(figsize=(12,6))
    plt.plot(eval_df["ds"], eval_df["y"], "o-", label="Gerçek")
    plt.plot(eval_df["ds"], eval_df["yhat"], "--", label="Tahmin")
    if {"pi80_lo","pi80_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi80_lo"], eval_df["pi80_hi"], alpha=0.25, label="PI 80%")
    if {"pi95_lo","pi95_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi95_lo"], eval_df["pi95_hi"], alpha=0.15, label="PI 95%")
    plt.title(title); plt.xlabel("Tarih"); plt.ylabel("Satış"); plt.legend(); plt.tight_layout()
    plt.savefig(savepath, dpi=160); plt.close()

# Make plots for the three variants on Full horizon
for variant in VARIANTS:
    path_csv = f"outputs/preds_Full_{variant.replace('-','').lower()}.csv"
    if os.path.exists(path_csv):
        dfp = pd.read_csv(path_csv, parse_dates=["ds"])
        plot_with_pi(dfp, f"{best_exog} • {variant} • Full Horizon", f"outputs/plot_full_{variant.replace('-','').lower()}.png")

print("\nArtifacts:")
print("- outputs/val_exog_selection.csv     (VAL kıyası ve ağırlıklar)")
print("- outputs/test_summary_selected_exog.csv  (TEST sonuç özeti + stockout olasılıkları)")
print("- outputs/preds_Full_*.csv , outputs/preds_Short3_*.csv  (tahmin + PI)")
print("- outputs/plot_full_*.png")
print("- outputs/feat_importance_*.csv  (model importances)")
print("- outputs/shap_importance_*.csv  (varsa SHAP)")


Importing plotly failed. Interactive plots will not work.
17:11:55 - cmdstanpy - INFO - Chain [1] start processing
17:11:55 - cmdstanpy - INFO - Chain [1] done processing



=== ROCV Best Params ===
RF : {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 1} | ROCV_MAE=71.55
XGB: {'n_estimators': 800, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0} | ROCV_MAE=59.43


/Users/baristezel/Desktop/veriler/path/to/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/baristezel/Desktop/veriler/path/to/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/baristezel/Desktop/veriler/path/to/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/baristezel/Desktop/veriler/path/to/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Like


=== Exog Selection (by VAL MAE of Y-ENS) ===
       Exog  VAL_MAE_RF  VAL_MAE_XGB  VAL_MAE_YENS  VAL_RMSE_YENS  VAL_MAPE_YENS     w_RF    w_XGB
   Ensemble  124.168646   126.224525    125.188146     139.117135     105.982818 0.504105 0.495895
 ML-Exog RF  112.138021   110.933634    111.532576     124.347148      84.627940 0.497300 0.502700
ML-Exog XGB  113.313021   108.419879    110.812459     123.732288      83.635409 0.488966 0.511034

Selected Exog: ML-Exog XGB | w_RF=0.49, w_XGB=0.51

=== TEST Summary (Selected Exog) ===
Horizon Selected_Exog Y-Variant       MAE      RMSE      MAPE     w_RF    w_XGB  P_stockout_3m  P_stockout_6m  E_T_stockout_mo
   Full   ML-Exog XGB        RF 56.846250 65.292464 49.066365      NaN      NaN        0.98250        1.00000         1.651250
   Full   ML-Exog XGB       XGB 55.802460 66.623825 49.623744      NaN      NaN        0.99625        1.00000         1.595000
   Full   ML-Exog XGB     Y-ENS 55.547385 65.751358 48.872797 0.488966 0.511034        